# Phase 3: tree vs vision vs both -- the fair comparison (Kaggle, resumable)

On our own v3 data UIA scored ~100%, but only because UIA chose the targets and wrote the instructions.
This test removes both advantages: 1,000 targets on 200 generated app pages, true boxes from the page itself,
five conditions checked on the rendered tree (`docs/research/gui-grounding-hypothesis.md`, H4-H6):

| | condition | | condition |
|---|---|---|---|
| C1 | named button | C4 | drawn on a canvas -- not in the tree |
| C2 | icon button with no name | C5 | same label in several panels |
| C3 | named, asked for in other words | | |

Arms: **tree** (name matching), **vlm** (our grounder, the dev-chosen adapter), **vlm_zero_shot**,
**fallback** (tree, else vlm), **snap** (vlm point moved into the tree element it lands in). All fixed before this run.

## Setup
Secret `HF_TOKEN` attached, **GPU T4 x2**, Internet **On**, **Input: the `compuse-compare` dataset only**.

## Run: Save Version -> "Save & Run All (Commit)" (~1.5 h). It stops itself if the grounder's first 20 answers don't parse.

In [ ]:
# Cell 1 -- settings. Runs before ANY torch/transformers import, on purpose.
import os
import time

SESSION_START = time.time()

RUN_MODE = "eval"                 # "eval" (all 1,000) or "smoke" (25 items)
HF_REPO_NAME = "computer-use-grounder"   # the adapter comes from it, results go back to it
CODE_REF = "8dac97df3fc4ce35546fa7a209d81dc7cd83a4db"           # exact commit of rudranaresh0201/computer-use

assert RUN_MODE in ("smoke", "eval"), RUN_MODE
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"RUN_MODE={RUN_MODE}  CODE_REF={CODE_REF}")

In [ ]:
# Cell 2 -- the GPU the model was trained and evaluated on.
import subprocess

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0, "No GPU attached. Session options -> Accelerator -> GPU T4 x2."
assert "T4" in smi.stdout, "Expected a Tesla T4 (Session options -> Accelerator -> GPU T4 x2)."

In [ ]:
# Cell 3 -- the token must work in THIS committed session and be able to write results.
import requests
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Could not read the HF_TOKEN secret. Add-ons -> Secrets: add HF_TOKEN, tick it for this notebook.") from e
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN secret is empty or not a Hugging Face token"
r = requests.get("https://huggingface.co/api/whoami-v2", headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=30)
assert r.status_code == 200, f"Hugging Face rejected the token (HTTP {r.status_code}): {r.text[:200]}"
me = r.json()
assert me.get("auth", {}).get("accessToken", {}).get("role") != "read", "HF_TOKEN is READ-only; results need write."
HF_REPO = f"{me['name']}/{HF_REPO_NAME}"
PREFIX = "compare-v1" if RUN_MODE == "eval" else f"compare-smoke-{CODE_REF[:7]}"
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"token OK for {me['name']}; adapter from {HF_REPO}, results to {HF_REPO}/{PREFIX}/")

In [ ]:
# Cell 4 -- the exact code version, and its pinned dependencies.
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/computer-use")
if not REPO_DIR.is_dir():
    !git clone https://github.com/rudranaresh0201/computer-use.git {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin
!git -C {REPO_DIR} checkout --quiet {CODE_REF}
!git -C {REPO_DIR} log -1 --format="evaluating with commit %H (%s)"
!{sys.executable} -m pip install --quiet -e "{REPO_DIR}[training]"
!{sys.executable} -m pip uninstall -y --quiet torchao

In [ ]:
# Cell 5 -- imports at the pinned versions, and the adapter our own dev split chose.
import importlib
import json

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()

import torch
import transformers

from computeruse.training.hub_checkpoint import HubSync

print(f"torch {torch.__version__} | transformers {transformers.__version__}")
assert transformers.__version__ == "5.0.0", "pinned transformers did not load -- restart and run from the top"
assert torch.cuda.is_available()

sync = HubSync(HF_REPO, token=HF_TOKEN)
print(f"Hub write check OK as {sync.preflight()}")
STORE_DIR = Path("/kaggle/working/compare_store")
assert sync.fetch(["eval/selection.json"], STORE_DIR), "no eval/selection.json -- run the v3 eval notebook first"
CHOSEN = json.loads((STORE_DIR / "eval" / "selection.json").read_text())["chosen"]
print(f"adapter chosen on our dev split: {CHOSEN}")

In [ ]:
# Cell 6 -- the frozen test set, exactly as its manifest describes it.
manifests = [p for p in Path("/kaggle/input").rglob("manifest.json") if (p.parent / "items.jsonl").exists()]
assert len(manifests) == 1, f"need exactly one compare test set under /kaggle/input, found {manifests}"
ROOT = manifests[0].parent
MANIFEST = json.loads(manifests[0].read_text())
ITEMS = [json.loads(line) for line in (ROOT / "items.jsonl").read_text().splitlines() if line.strip()]
VIEWPORT = tuple(MANIFEST["viewport"])
print(f"test set {ROOT}: {len(ITEMS)} items, by condition {MANIFEST['by_condition']}, pages from seed {MANIFEST['first_seed']}")
assert len(ITEMS) == MANIFEST["items"] and MANIFEST["first_seed"] >= 1000, "not the frozen test set"
if RUN_MODE == "smoke":
    ITEMS = [i for c in ("C1", "C2", "C3", "C4", "C5") for i in [x for x in ITEMS if x["condition"] == c][:5]]
    print(f"smoke: {len(ITEMS)} items")

In [ ]:
# Cell 7 -- base model + the one chosen adapter.
from transformers import AutoModelForImageTextToText, AutoProcessor

from computeruse.eval.ablation import adapter_path
from computeruse.eval.resumable import ResultStore
from computeruse.eval.vlm_grounder import MODEL_ID, ZERO_SHOT, arm_active, load_adapters
from computeruse.training.prepare_dataset import TrainingExample
from computeruse.training.train_lora import resolve_precision, torch_dtype_for

ADAPTER_ROOT = Path("/kaggle/working/adapters")
folder = "final" if CHOSEN == "final" else f"adapters/{CHOSEN}"
assert sync.fetch([f"{folder}/adapter_*"], ADAPTER_ROOT), f"adapter {CHOSEN} not found on the Hub"

precision = resolve_precision()
processor = AutoProcessor.from_pretrained(MODEL_ID)
base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch_dtype_for(precision), attn_implementation="sdpa"
).to("cuda")
model = load_adapters(base, {CHOSEN: adapter_path(CHOSEN, ADAPTER_ROOT)})
del base  # the plain base model is now only arm_active(model, ZERO_SHOT)
print(f"{MODEL_ID} in {precision} + {CHOSEN}; GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")

store = ResultStore(sync, PREFIX, STORE_DIR, flush_every_seconds=300)
print(f"results already on the Hub under {PREFIX}/: {store.pull()}")

In [ ]:
# Cell 8 -- the grounder on every item: fine-tuned first (with a stop-if-broken check), then zero-shot.
from computeruse.eval import compare
from computeruse.eval.resumable import score_arm
from computeruse.eval.vlm_grounder import summarize_diagnostics

ITEM_BY_ID = {i["id"]: i for i in ITEMS}
EXAMPLES = [TrainingExample(i["id"], i["image"], i["instruction"], "", i["condition"], i["condition"]) for i in ITEMS]


def score(example):
    return compare.score_vlm(model, processor, ITEM_BY_ID[example.id], ROOT)


with arm_active(model, CHOSEN), torch.no_grad():
    first = score_arm(store, CHOSEN, EXAMPLES[:20], score)
parse_rate = summarize_diagnostics(first)["parse_rate"]
print(f"\nsanity check -- {CHOSEN} parse rate on the first 20: {parse_rate:.0%}")
for r in first[:5]:
    print(f"  {r.split} {ITEM_BY_ID[r.example_id]['instruction'][:40]!r:<44} -> {r.predicted_text!r} hit={r.hit}")
assert parse_rate >= 0.8, "the grounder's answers are not parsing -- stopping before spending the GPU budget"

VLM = {}
for arm in (CHOSEN, ZERO_SHOT):
    with arm_active(model, arm), torch.no_grad():
        VLM[arm] = {r.example_id: r for r in score_arm(store, arm, EXAMPLES, score)}

In [ ]:
# Cell 9 -- every arm on every item, the table, and H4-H6.
from computeruse.web.browser import parse_snapshot

trees = {}
SCORED = []
for item in ITEMS:
    if item["tree"] not in trees:
        trees[item["tree"]] = parse_snapshot((ROOT / item["tree"]).read_text(encoding="utf-8"))
    SCORED.append(compare.score_item(item, trees[item["tree"]], VLM[CHOSEN][item["id"]].predicted_point,
                                     VLM[ZERO_SHOT][item["id"]].predicted_point, VIEWPORT))
REPORT = compare.report(SCORED)
REPORT.update({"adapter": CHOSEN, "code_ref": CODE_REF, "items": len(ITEMS),
               "manifest": {k: v for k, v in MANIFEST.items() if k != "dropped"}})
Path("/kaggle/working/compare_report.json").write_text(json.dumps(REPORT, indent=2))
store.push_json("report.json", REPORT)
store.push_json("scored.json", SCORED)

columns = list(compare.CONDITIONS) + ["mean"]
print(f"\nclick accuracy by condition (vlm = {CHOSEN}), [95% CI]")
print(f"{'arm':<15}" + "".join(f"{c:>17}" for c in columns))
for arm in compare.ARMS:
    row = REPORT["table"][arm]
    cells = []
    for c in columns:
        cell = row[c]
        if cell.get("accuracy") is None:
            cells.append("-")
        elif c == "mean":
            cells.append(f"{cell['accuracy']:.1%}")
        else:
            lo, hi = cell["ci95"]
            cells.append(f"{cell['accuracy']:.0%} [{lo:.0%},{hi:.0%}]")
    print(f"{arm:<15}" + "".join(f"{x:>17}" for x in cells))

print("\nhypotheses (paired difference, 95% CI):")
for name, d in REPORT["hypotheses"].items():
    lo, hi = d["ci95"]
    word = "above 0" if lo > 0 else "below 0" if hi < 0 else "includes 0"
    print(f"  {name:<36} {d['diff']:+.1%} [{lo:+.1%}, {hi:+.1%}]  n={d['n']}  -> interval {word}")

print("\n" + "=" * 70)
hours = (time.time() - SESSION_START) / 3600
print(("COMPARE SMOKE PASSED" if RUN_MODE == "smoke" else "COMPARE FINISHED")
      + f" in {hours:.1f} h. Report: {HF_REPO}/{PREFIX}/report.json")
print("=" * 70)